In [49]:
# Data path for R library 

# /var/folders/lt/fs9cxhhn7_v8qvhkq56m6zh80000gn/T//RtmpQEZlmN/downloaded_packages

In [59]:
!playwright install chromium

In [50]:
import requests
from bs4 import BeautifulSoup
import json

# Use the standard readable URL
url = "https://www.fotmob.com/players/422685/bruno-fernandes"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Next.js stores all the page data in this specific tag
    next_data_script = soup.find('script', id='__NEXT_DATA__')
    
    if next_data_script:
        # Load the text inside the script tag as a python dictionary
        json_data = json.loads(next_data_script.string)
        
        # The data you need is nested inside the props
        # Let's print the keys to help you navigate it
        print("Data extracted successfully. Top level keys:")
        print(json_data.keys())
        
        # To get you started, the core stats are usually under:
        # page_props = json_data['props']['pageProps']
    else:
        print("Could not find the __NEXT_DATA__ script tag.")
else:
    print(f"Request failed. Status code: {response.status_code}")

Data extracted successfully. Top level keys:
dict_keys(['props', 'page', 'query', 'buildId', 'isFallback', 'isExperimentalCompile', 'dynamicIds', 'gssp', 'appGip', 'scriptLoader'])


In [51]:
# Dig into the 'props' key
page_props = json_data['props']['pageProps']

print("Keys inside pageProps:")
print(page_props.keys())

# In many Next.js sites, the actual API response is cached inside a 'fallback' dictionary
if 'fallback' in page_props:
    fallback_data = page_props['fallback']
    
    # The keys in 'fallback' are usually the API URLs themselves
    # Let's get the first key and look inside
    api_url_key = list(fallback_data.keys())[0]
    player_data = fallback_data[api_url_key]
    
    print("\nSuccess! Here are the keys for the actual player data:")
    if isinstance(player_data, dict):
        print(player_data.keys())
else:
    # If no fallback, it might be directly in pageProps
    print("\nNo fallback found, check pageProps directly for stats/matches.")

Keys inside pageProps:
dict_keys(['data', 'fallback', 'translations'])

Success! Here are the keys for the actual player data:
dict_keys(['id', 'name', 'birthDate', 'contractEnd', 'isCoach', 'isCaptain', 'gender', 'primaryTeam', 'positionDescription', 'injuryInformation', 'internationalDuty', 'playerInformation', 'mainLeague', 'trophies', 'recentMatches', 'careerHistory', 'traits', 'meta', 'coachStats', 'statSeasons', 'firstSeasonStats', 'status', 'marketValues', 'relatedLinksData', 'nextMatch', 'dataProvider', 'ssr'])


In [52]:
# Extract the list of recent matches
recent_matches = player_data.get('recentMatches', [])

if recent_matches:
    print(f"Found {len(recent_matches)} recent matches!")
    
    # Grab the very first match in the list to inspect its structure
    first_match = recent_matches[0]
    
    print("\nKeys inside a single match object:")
    print(first_match.keys())
    
    # Let's print out some of the juicy details if they exist
    print("\nSample Match Data:")
    print(f"Match ID: {first_match.get('id', 'Not found')}")
    print(f"Opponent: {first_match.get('opponent', {}).get('name', 'Not found')}")
    print(f"Player Rating: {first_match.get('rating', 'Not found')}")
    
else:
    print("No recent matches found in this dataset.")

Found 56 recent matches!

Keys inside a single match object:
dict_keys(['teamId', 'teamName', 'opponentTeamId', 'opponentTeamName', 'isHomeTeam', 'id', 'matchDate', 'matchPageUrl', 'leagueId', 'leagueName', 'stage', 'homeScore', 'awayScore', 'minutesPlayed', 'goals', 'assists', 'yellowCards', 'redCards', 'ratingProps', 'playerOfTheMatch', 'onBench'])

Sample Match Data:
Match ID: 4813697
Opponent: Not found
Player Rating: Not found


In [53]:
import requests
from bs4 import BeautifulSoup
import json

match_id = "4813697"

# Use the standard front-facing web URL instead of the dead API endpoint
url = f"https://www.fotmob.com/match/{match_id}"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'html.parser')
    next_data_script = soup.find('script', id='__NEXT_DATA__')
    
    if next_data_script:
        json_data = json.loads(next_data_script.string)
        page_props = json_data['props']['pageProps']
        
        print("Success, bypassed the 404. Keys inside the Next.js Match pageProps:")
        print(page_props.keys())
        
        # Digging into the fallback cache where the match stats usually live
        if 'fallback' in page_props:
            fallback_data = page_props['fallback']
            
            # Grab the first key in the fallback dictionary
            api_url_key = list(fallback_data.keys())[0]
            match_details = fallback_data[api_url_key]
            
            print("\nKeys inside the actual match details:")
            if isinstance(match_details, dict):
                print(match_details.keys())
                
                # Check if the specific lineup data is nested in here
                if 'content' in match_details and 'lineup' in match_details['content']:
                    print("\nFound the lineup data! Keys inside lineup:")
                    print(match_details['content']['lineup'].keys())
    else:
        print("Could not find the __NEXT_DATA__ script tag.")
else:
    print(f"Request failed. Status code: {response.status_code}")

Success, bypassed the 404. Keys inside the Next.js Match pageProps:
dict_keys(['general', 'header', 'nav', 'ongoing', 'hasPendingVAR', 'content', 'seo', 'fetchingLeagueData', 'ssr', 'fallback', 'translations'])

Keys inside the actual match details:
dict_keys(['matches'])


In [54]:
import pandas as pd

# Using your exact file path and fixing the Dtype warning
df = pd.read_csv('/Users/schoudhry/Desktop/mind-overnight/code/epl_match_details.csv', low_memory=False)

# Print all the column names so we can see the exact labels
print("Available columns in the dataset:")
columns = df.columns.tolist()

# Grouping them by 5 so it is easy to read
for i in range(0, len(columns), 5):
    print(columns[i:i+5])

Available columns in the dataset:
['match_id', 'match_round', 'league_id', 'league_name', 'league_round_name']
['parent_league_id', 'parent_league_season', 'match_time_utc', 'home_team_id', 'home_team']
['home_team_color', 'away_team_id', 'away_team', 'away_team_color', 'id']
['event_type', 'team_id', 'player_id', 'player_name', 'x']
['y', 'min', 'min_added', 'is_blocked', 'is_on_target']
['goal_crossed_y', 'expected_goals', 'expected_goals_on_target', 'shot_type', 'situation']
['period', 'is_own_goal', 'on_goal_shot_x', 'on_goal_shot_y', 'on_goal_shot_zoom_ratio']
['first_name', 'last_name', 'team_color', 'short_name', 'blocked_x']
['blocked_y', 'goal_crossed_z', 'full_name', 'is_saved_off_line', 'is_from_inside_box']


In [55]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import json

# 1. Load the match IDs
df = pd.read_csv('/Users/schoudhry/Desktop/mind-overnight/code/epl_match_details.csv', low_memory=False)
match_ids = df['match_id'].unique().tolist()

test_match_id = match_ids[0]
url = f"https://www.fotmob.com/match/{test_match_id}"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')
next_data_script = soup.find('script', id='__NEXT_DATA__')

json_data = json.loads(next_data_script.string)

# --- THE BULLETPROOF FIX ---
# This recursive function scans the entire JSON tree to find the lineup array

def find_lineups(data):
    # We are looking for a list of teams that contains 'teamName' and 'players'
    if isinstance(data, list) and len(data) > 0 and isinstance(data[0], dict):
        if 'teamName' in data[0] and 'players' in data[0]:
            return data
            
    if isinstance(data, dict):
        for key, value in data.items():
            result = find_lineups(value)
            if result is not None:
                return result
    elif isinstance(data, list):
        for item in data:
            result = find_lineups(item)
            if result is not None:
                return result
    return None

lineups = find_lineups(json_data)

if not lineups:
    raise Exception("Could not find lineup data anywhere in the Next.js payload.")

player_stats = []

# Loop through the extracted lineups
for team in lineups:
    team_name = team.get('teamName', 'Unknown')
    
    if 'players' in team:
        for position_line in team['players']:
            for player in position_line:
                
                # Safely extract rating
                rating = None
                if 'rating' in player and 'num' in player['rating']:
                    rating = player['rating']['num']
                
                # Safely extract name
                player_name = player.get('name', 'Unknown')
                if isinstance(player_name, dict) and 'fullName' in player_name:
                    player_name = player_name['fullName']
                    
                player_stats.append({
                    'Match_ID': test_match_id,
                    'Team': team_name,
                    'Player': player_name,
                    'Role': player.get('role', 'Unknown'),
                    'Rating': rating
                })

ratings_df = pd.DataFrame(player_stats)
print(ratings_df.head(20))

Exception: Could not find lineup data anywhere in the Next.js payload.